In [1]:
from dataclasses import dataclass
from enum import Enum
import random
import time
import uuid
from typing import Any, Callable, Dict, List, Optional


# ==========================================
# 1. 归一化异常体系与 ToolResult 契约
# ==========================================
class ErrorKind(Enum):
    RETRYABLE = "retryable"
    NON_RETRYABLE = "non_retryable"
    RATE_LIMITED = "rate_limited"
    TIMEOUT = "timeout"
    OVERLOADED = "overloaded"
    CANCELLED = "cancelled"
    INTERNAL = "internal"


@dataclass
class ToolError:
    kind: ErrorKind
    message: str
    retryable: bool
    status_code: Optional[int] = None
    retry_after: Optional[float] = None


@dataclass
class ToolResult:
    ok: bool
    tool_name: str
    data: Any = None
    error: Optional[ToolError] = None
    attempts: int = 0
    latency_ms: float = 0.0


def normalize_error(
    status_code: Optional[int] = None,
    exc: Optional[Exception] = None,
    retry_after: Optional[float] = None,
    is_bulkhead_rejected: bool = False,
    is_circuit_open: bool = False,
) -> ToolError:
    """将 Raw Error 归一化映射为标准 ToolError"""
    if is_circuit_open:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="CircuitBreaker is OPEN",
            retryable=False,
        )

    if is_bulkhead_rejected:
        return ToolError(
            kind=ErrorKind.OVERLOADED,
            message="Bulkhead capacity full",
            retryable=False,
            status_code=503,
        )

    if exc is not None:
        if isinstance(exc, TimeoutError):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=str(exc) or "Request execution timed out",
                retryable=True,
            )
        return ToolError(
            kind=ErrorKind.INTERNAL,
            message=f"Internal exception: {type(exc).__name__} - {str(exc)}",
            retryable=False,
        )

    if status_code is not None:
        if status_code == 429:
            return ToolError(
                kind=ErrorKind.RATE_LIMITED,
                message="HTTP 429 Too Many Requests",
                retryable=True,
                status_code=429,
                retry_after=retry_after or 0.1,
            )
        if status_code in (500, 502, 503, 504):
            return ToolError(
                kind=ErrorKind.RETRYABLE,
                message=f"HTTP {status_code} Server Error",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (408,):
            return ToolError(
                kind=ErrorKind.TIMEOUT,
                message=f"HTTP {status_code} Request Timeout",
                retryable=True,
                status_code=status_code,
            )
        if status_code in (400, 401, 403, 404, 422):
            return ToolError(
                kind=ErrorKind.NON_RETRYABLE,
                message=f"HTTP {status_code} Client Error",
                retryable=False,
                status_code=status_code,
            )

    return ToolError(
        kind=ErrorKind.INTERNAL,
        message=f"Unknown raw error (status={status_code})",
        retryable=False,
        status_code=status_code,
    )


# ==========================================
# 2. Policy 决策引擎 (PolicyEngine)
# ==========================================
class PolicyAction(Enum):
    RETRY = "retry"
    FAIL = "fail"
    FALLBACK = "fallback"
    CANCEL = "cancel"
    RATE_LIMIT_WAIT = "rate_limit_wait"
    DEADLINE_EXCEEDED = "deadline_exceeded"


@dataclass
class PolicyDecision:
    action: PolicyAction
    delay: float = 0.0
    reason: str = ""


class PolicyEngine:
    """无状态纯函数决策引擎：不引发任何 Side Effect (不调工具/不加锁/不写状态)"""

    @staticmethod
    def decide(
        error: ToolError,
        retry_count: int,
        max_retries: int,
        base_delay: float,
        remaining_budget: float,
        request_timeout: float,
    ) -> PolicyDecision:
        # 1. 校验错误是否具备重试资格
        if not error.retryable:
            return PolicyDecision(
                action=PolicyAction.FAIL,
                reason=f"Non-retryable error kind: {error.kind.value}",
            )

        # 2. 校验重试次数预算
        if retry_count >= max_retries:
            return PolicyDecision(
                action=PolicyAction.FAIL,
                reason=f"Max retries reached ({max_retries})",
            )

        # 3. 计算延迟时间 Delay (针对 429 RATE_LIMITED 与 普通 Backoff 做区分)
        if error.kind == ErrorKind.RATE_LIMITED:
            computed_delay = error.retry_after if error.retry_after is not None else 0.1
            action = PolicyAction.RATE_LIMIT_WAIT
        else:
            backoff = base_delay * (2 ** retry_count)
            jitter = random.uniform(0.0, 0.01)
            computed_delay = backoff + jitter
            action = PolicyAction.RETRY

        # 4. 校验 Deadline 时间预算
        required_budget = computed_delay + request_timeout
        if remaining_budget < required_budget:
            return PolicyDecision(
                action=PolicyAction.DEADLINE_EXCEEDED,
                delay=0.0,
                reason=f"Insufficient remaining budget ({remaining_budget:.3f}s < required {required_budget:.3f}s)",
            )

        return PolicyDecision(
            action=action,
            delay=computed_delay,
            reason=f"Allowed {action.value} after {computed_delay:.3f}s delay",
        )


# ==========================================
# 3. 基础组件与配置 (CircuitBreaker & Bulkhead & Registry)
# ==========================================
class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, cooldown: float = 5.0):
        self.state: str = "CLOSED"
        self.failure_count: int = 0
        self.failure_threshold: int = failure_threshold
        self.cooldown: float = cooldown
        self.opened_at: Optional[float] = None

    def can_call(self) -> bool:
        now = time.time()
        if self.state == "OPEN":
            if self.opened_at and (now - self.opened_at >= self.cooldown):
                self.state = "HALF_OPEN"
                return True
            return False
        return True

    def record_success(self):
        self.failure_count = 0
        self.state = "CLOSED"

    def record_failure(self):
        self.failure_count += 1
        if self.state == "HALF_OPEN" or self.failure_count >= self.failure_threshold:
            self.state = "OPEN"
            self.opened_at = time.time()


class Bulkhead:
    def __init__(self, capacity: int):
        self.capacity: int = capacity
        self.in_flight: int = 0

    def try_acquire(self) -> bool:
        if self.in_flight < self.capacity:
            self.in_flight += 1
            return True
        return False

    def release(self) -> None:
        if self.in_flight > 0:
            self.in_flight -= 1


@dataclass
class ToolConfig:
    name: str
    max_retries: int
    base_delay: float
    timeout: float
    idempotency_required: bool
    breaker: CircuitBreaker
    bulkhead: Bulkhead


class ToolRegistry:
    def __init__(self):
        self.configs: Dict[str, ToolConfig] = {}
        self.funcs: Dict[str, Callable] = {}

    def register(self, config: ToolConfig, func: Callable):
        self.configs[config.name] = config
        self.funcs[config.name] = func

    def get_config(self, name: str) -> ToolConfig:
        return self.configs[name]

    def get_func(self, name: str) -> Callable:
        return self.funcs[name]


# ==========================================
# 4. 只负责执行 PolicyDecision 的 ToolRuntime
# ==========================================
class ToolRuntime:
    def __init__(self, registry: ToolRegistry):
        self.registry = registry

    def execute(self, tool_name: str, args: Dict[str, Any], deadline: Optional[float] = None) -> ToolResult:
        start_time = time.perf_counter()

        config = self.registry.get_config(tool_name)
        tool_fn = self.registry.get_func(tool_name)

        if config.idempotency_required and "idempotency_key" not in args:
            args["idempotency_key"] = f"idempotent-{uuid.uuid4().hex[:8]}"

        if deadline is None:
            deadline = time.time() + 10.0

        # Breaker Gate 检查
        if not config.breaker.can_call():
            elapsed_ms = (time.perf_counter() - start_time) * 1000
            return ToolResult(
                ok=False,
                tool_name=tool_name,
                data=None,
                error=normalize_error(is_circuit_open=True),
                attempts=0,
                latency_ms=elapsed_ms,
            )

        retry_count = 0

        while True:
            # Bulkhead Acquire 检查
            if not config.bulkhead.try_acquire():
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=None,
                    error=normalize_error(is_bulkhead_rejected=True),
                    attempts=retry_count,
                    latency_ms=elapsed_ms,
                )

            raw_result = None
            caught_exc = None
            try:
                raw_result = tool_fn(args)
            except Exception as e:
                caught_exc = e
            finally:
                config.bulkhead.release()

            # 解析 Status Code
            status_code = None
            retry_after = None
            if isinstance(raw_result, dict):
                status_code = raw_result.get("status")
                retry_after = raw_result.get("retry_after")

            # 错误归一化 (Raw Error -> ToolError)
            if caught_exc is not None or (status_code and status_code != 200):
                tool_error = normalize_error(
                    status_code=status_code,
                    exc=caught_exc,
                    retry_after=retry_after,
                )
            else:
                tool_error = None

            # 1. 调用成功分支
            if tool_error is None:
                config.breaker.record_success()
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=True,
                    tool_name=tool_name,
                    data=raw_result,
                    error=None,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            # 2. 失败分支：通知 CircuitBreaker
            config.breaker.record_failure()

            # ----------------------------------------------------
            # 核心改进：把决策交由 PolicyEngine 判定，Runtime 只机械执行
            # ----------------------------------------------------
            remaining_budget = deadline - time.time()
            decision = PolicyEngine.decide(
                error=tool_error,
                retry_count=retry_count,
                max_retries=config.max_retries,
                base_delay=config.base_delay,
                remaining_budget=remaining_budget,
                request_timeout=config.timeout,
            )

            # 方案 A: 决策为 FAIL
            if decision.action == PolicyAction.FAIL:
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=raw_result,
                    error=tool_error,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            # 方案 B: 决策为 DEADLINE_EXCEEDED
            if decision.action == PolicyAction.DEADLINE_EXCEEDED:
                elapsed_ms = (time.perf_counter() - start_time) * 1000
                deadline_err = ToolError(
                    kind=ErrorKind.TIMEOUT,
                    message=f"Deadline Exceeded: {decision.reason}",
                    retryable=False,
                )
                return ToolResult(
                    ok=False,
                    tool_name=tool_name,
                    data=None,
                    error=deadline_err,
                    attempts=retry_count + 1,
                    latency_ms=elapsed_ms,
                )

            # 方案 C: 决策为 RETRY 或 RATE_LIMIT_WAIT
            if decision.action in (PolicyAction.RETRY, PolicyAction.RATE_LIMIT_WAIT):
                time.sleep(decision.delay)
                retry_count += 1


# ==========================================
# 5. 4 个核心 Scenario 验证
# ==========================================
if __name__ == "__main__":
    err_503 = ToolError(kind=ErrorKind.RETRYABLE, message="503 Error", retryable=True)

    print("=== Scenario 1: 503 + budget 足够 -> RETRY ===")
    dec1 = PolicyEngine.decide(
        error=err_503,
        retry_count=0,
        max_retries=2,
        base_delay=0.1,
        remaining_budget=5.0,
        request_timeout=1.0,
    )
    print(f"Action: {dec1.action} (Expected: PolicyAction.RETRY)")
    print(f"Delay: {dec1.delay:.3f}s | Reason: {dec1.reason}\n")

    print("=== Scenario 2: 503 + retry 用完 -> FAIL ===")
    dec2 = PolicyEngine.decide(
        error=err_503,
        retry_count=2,
        max_retries=2,
        base_delay=0.1,
        remaining_budget=5.0,
        request_timeout=1.0,
    )
    print(f"Action: {dec2.action} (Expected: PolicyAction.FAIL)")
    print(f"Reason: {dec2.reason}\n")

    print("=== Scenario 3: 429 + retry_after=3 -> RATE_LIMIT_WAIT, delay=3 ===")
    err_429 = ToolError(kind=ErrorKind.RATE_LIMITED, message="429 Limit", retryable=True, retry_after=3.0)
    dec3 = PolicyEngine.decide(
        error=err_429,
        retry_count=0,
        max_retries=2,
        base_delay=0.1,
        remaining_budget=10.0,
        request_timeout=1.0,
    )
    print(f"Action: {dec3.action} (Expected: PolicyAction.RATE_LIMIT_WAIT)")
    print(f"Delay: {dec3.delay}s (Expected: 3.0)")
    print(f"Reason: {dec3.reason}\n")

    print("=== Scenario 4: 503 + deadline 不够 -> DEADLINE_EXCEEDED ===")
    dec4 = PolicyEngine.decide(
        error=err_503,
        retry_count=0,
        max_retries=2,
        base_delay=0.5,
        remaining_budget=0.8,  # 剩余预算小于 (0.5s backoff + 1.0s timeout)
        request_timeout=1.0,
    )
    print(f"Action: {dec4.action} (Expected: PolicyAction.DEADLINE_EXCEEDED)")
    print(f"Reason: {dec4.reason}\n")

=== Scenario 1: 503 + budget 足够 -> RETRY ===
Action: PolicyAction.RETRY (Expected: PolicyAction.RETRY)
Delay: 0.101s | Reason: Allowed retry after 0.101s delay

=== Scenario 2: 503 + retry 用完 -> FAIL ===
Action: PolicyAction.FAIL (Expected: PolicyAction.FAIL)
Reason: Max retries reached (2)

=== Scenario 3: 429 + retry_after=3 -> RATE_LIMIT_WAIT, delay=3 ===
Action: PolicyAction.RATE_LIMIT_WAIT (Expected: PolicyAction.RATE_LIMIT_WAIT)
Delay: 3.0s (Expected: 3.0)
Reason: Allowed rate_limit_wait after 3.000s delay

=== Scenario 4: 503 + deadline 不够 -> DEADLINE_EXCEEDED ===
Action: PolicyAction.DEADLINE_EXCEEDED (Expected: PolicyAction.DEADLINE_EXCEEDED)
Reason: Insufficient remaining budget (0.800s < required 1.500s)

